# Material opcional de la semana 3: fine-tuning con LoRA sobre Qwen2.5-0.5B

No forma parte de la sesión ni de la práctica. Sirve para quien necesite afinar un modelo abierto en su tesis y quiera ver el mecanismo completo de la slide 20 ("Fine-tuning: la familia completa") en un notebook que cabe en Colab gratuito. En Colab conviene activar la GPU (Entorno de ejecución → Cambiar tipo de entorno → T4); en CPU también corre, más lento.

Qué hace:

1. Construye un dataset pequeño de instrucciones de la mesa de soporte de Facturio (clasificación de tickets en el formato de chat de Qwen).
2. Congela el modelo base y agrega adaptadores LoRA a las matrices de atención.
3. Entrena unos cientos de pasos con next-token prediction sobre la respuesta.
4. Compara base, base + LoRA e instruct sobre tickets que no vio en el entrenamiento, con la misma métrica de la semana 4 (accuracy de categoría).
5. Guarda el adaptador (unos MB) por separado del modelo.

Cuándo tiene sentido LoRA: comportamiento o formato muy específico, en volumen alto, con dataset propio de calidad. Rara vez para agregar conocimiento. Antes de afinar, medir que prompting (semana 4) y RAG (semanas 5 a 7) no bastan.

In [1]:
# En Colab: torch y transformers ya vienen; se instalan peft, datasets y accelerate.
%pip install -q peft datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import argparse
import json
import random
import time
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

BASE = "Qwen/Qwen2.5-0.5B"
INSTRUCT = "Qwen/Qwen2.5-0.5B-Instruct"
CATEGORIAS = ["facturacion", "acceso", "error_tecnico", "solicitud_funcion", "otro"]
SISTEMA = ("Eres el clasificador de tickets de la mesa de soporte de Facturio. "
           "Responde únicamente con una de estas categorías: " + ", ".join(CATEGORIAS) + ".")

random.seed(0); torch.manual_seed(0)

/Users/junbarrs/clases_maestria/Arquitectura de Software para sistemas inteligentes/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. El dataset

Treinta tickets etiquetados para entrenar y diez distintos para evaluar. En un caso real el dataset sale de tickets históricos etiquetados por el equipo de soporte; la calidad de esas etiquetas es lo que el modelo va a aprender, con sus aciertos y sus errores.

Cada ejemplo se convierte al chat template de Qwen (semana 3, slides 23 y 24). La pérdida se calcula solo sobre los tokens de la respuesta: los del prompt llevan etiqueta `-100` y el entrenador los ignora. Así el modelo aprende a producir la categoría, no a repetir la instrucción.

In [3]:
# ---------------------------------------------------------------- dataset
# Tickets escritos para el ejercicio, distintos de los 50 de shared/datasets.
# En un caso real el dataset sale de tickets históricos etiquetados por el equipo.
ENTRENAMIENTO = [
    ("Me cobraron la mensualidad dos veces en agosto.", "facturacion"),
    ("Quiero cambiar el RFC que aparece en mis facturas.", "facturacion"),
    ("¿Pueden enviarme la factura de julio con uso de CFDI G03?", "facturacion"),
    ("El cargo de este mes no coincide con mi plan.", "facturacion"),
    ("Necesito cancelar la suscripción antes del siguiente cobro.", "facturacion"),
    ("Quiero pagar por transferencia en vez de tarjeta.", "facturacion"),
    ("Mi contraseña nueva no funciona y no puedo entrar.", "acceso"),
    ("La cuenta quedó bloqueada por intentos fallidos.", "acceso"),
    ("No me llega el código de dos factores por SMS.", "acceso"),
    ("Perdí el correo con el que me registré, ¿cómo recupero la cuenta?", "acceso"),
    ("Mi sesión se cierra sola cada pocos minutos.", "acceso"),
    ("No puedo entrar desde la red de la oficina, desde casa sí.", "acceso"),
    ("Al exportar a PDF las gráficas salen vacías.", "error_tecnico"),
    ("La aplicación se cierra al abrir la pestaña de reportes.", "error_tecnico"),
    ("El botón de guardar no responde en Safari.", "error_tecnico"),
    ("El buscador no encuentra registros con acentos.", "error_tecnico"),
    ("Los totales del dashboard no cuadran con el reporte descargado.", "error_tecnico"),
    ("El correo de invitación a un usuario nuevo nunca llega.", "error_tecnico"),
    ("Estaría bien poder exportar a Excel además de PDF.", "solicitud_funcion"),
    ("¿Podrían agregar modo oscuro?", "solicitud_funcion"),
    ("Me gustaría silenciar notificaciones por horario.", "solicitud_funcion"),
    ("Sugerencia: permitir adjuntos de más de 25 MB.", "solicitud_funcion"),
    ("Sería útil una API para integrar con nuestro ERP.", "solicitud_funcion"),
    ("Propongo sincronizar vencimientos con Google Calendar.", "solicitud_funcion"),
    ("¿Tienen oficina en Guadalajara?", "otro"),
    ("Gracias, el soporte de la semana pasada fue excelente.", "otro"),
    ("¿Cuál es el horario de atención telefónica?", "otro"),
    ("¿Cómo cambio el idioma de la interfaz?", "otro"),
    ("¿Me pueden mandar el contrato de servicio en PDF?", "otro"),
    ("¿Puedo tener dos administradores en la misma cuenta?", "otro"),
]
PRUEBA = [
    ("Aparece un cargo de 899 pesos y mi plan es el básico.", "facturacion"),
    ("La factura de septiembre tiene mal la razón social.", "facturacion"),
    ("Dice que mi contraseña es incorrecta aunque la acabo de cambiar.", "acceso"),
    ("Me pide un código de autenticación que nunca recibo.", "acceso"),
    ("La pantalla de carga se queda en 99 % y no termina.", "error_tecnico"),
    ("El enlace del manual devuelve error 404.", "error_tecnico"),
    ("Quisiera un resumen semanal por correo con la actividad del equipo.", "solicitud_funcion"),
    ("¿Pueden agregar colores a las etiquetas?", "solicitud_funcion"),
    ("¿Tienen descuento para escuelas?", "otro"),
    ("Quiero eliminar mi cuenta y todos mis datos.", "otro"),
]


def mensajes(texto: str, categoria: str | None = None) -> list[dict]:
    m = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": f"Ticket: {texto}"}]
    if categoria is not None:
        m.append({"role": "assistant", "content": categoria})
    return m


def preparar_dataset(tok) -> Dataset:
    """Cada ejemplo se tokeniza con el chat template; la pérdida solo se calcula
    sobre los tokens de la respuesta (los del prompt llevan etiqueta -100)."""
    filas = []
    for texto, cat in ENTRENAMIENTO:
        prompt = tok.apply_chat_template(mensajes(texto), add_generation_prompt=True, tokenize=False)
        completo = prompt + cat + tok.eos_token
        ids_prompt = tok(prompt, add_special_tokens=False).input_ids
        ids = tok(completo, add_special_tokens=False).input_ids
        labels = [-100] * len(ids_prompt) + ids[len(ids_prompt):]
        filas.append({"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels})
    return Dataset.from_list(filas)


def collate(batch, pad_id: int):
    n = max(len(b["input_ids"]) for b in batch)
    def rellenar(v, valor): return v + [valor] * (n - len(v))
    return {
        "input_ids": torch.tensor([rellenar(b["input_ids"], pad_id) for b in batch]),
        "attention_mask": torch.tensor([rellenar(b["attention_mask"], 0) for b in batch]),
        "labels": torch.tensor([rellenar(b["labels"], -100) for b in batch]),
    }

## 2. La evaluación

La misma métrica de la semana 4: accuracy de categoría sobre los diez tickets de prueba. El base se evalúa con un prompt plano (sin template, porque no lo conoce) y los otros dos con el template.

In [4]:
# ---------------------------------------------------------------- evaluación
@torch.no_grad()
def clasificar(modelo, tok, texto: str, usar_template: bool = True) -> str:
    if usar_template:
        prompt = tok.apply_chat_template(mensajes(texto), add_generation_prompt=True, tokenize=False)
    else:  # el base sin template: texto plano
        prompt = f"{SISTEMA}\nTicket: {texto}\nCategoría:"
    ids = tok(prompt, return_tensors="pt").input_ids.to(modelo.device)
    out = modelo.generate(ids, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()


def evaluar(nombre: str, modelo, tok, usar_template: bool = True) -> float:
    aciertos = 0
    for texto, esperado in PRUEBA:
        salida = clasificar(modelo, tok, texto, usar_template)
        ok = salida.lower().startswith(esperado)
        aciertos += ok
        print(f"  [{nombre}] {'ok ' if ok else 'X  '} {esperado:18s} <- {salida[:40]!r}")
    acc = aciertos / len(PRUEBA)
    print(f"  [{nombre}] accuracy {acc:.2f}\n")
    return acc

## 3. Línea base: el modelo base con un prompt plano

Predicción antes de ejecutar: ¿cuántos de los diez tickets clasificará bien un modelo base de 500 millones de parámetros sin ejemplos?

In [5]:
dispositivo = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if dispositivo == "cuda" else torch.float32
print(f"dispositivo: {dispositivo}")

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype).to(dispositivo).eval()
print("Base, prompt plano:")
acc_base = evaluar("base", base, tok, usar_template=False)

dispositivo: mps


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:00<00:59,  4.90it/s]

Loading weights:  92%|█████████▏| 267/290 [00:00<00:00, 1083.89it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 913.83it/s] 

Base, prompt plano:


  [base] X   facturacion        <- 'error_tecnico\nTicket'
  [base] X   facturacion        <- 'error_tecnico\nTicket'
  [base] X   acceso             <- 'error_tecnico\nTicket'


  [base] X   acceso             <- 'error_tecnico\nTicket'
  [base] ok  error_tecnico      <- 'error_tecnico\nTicket'


  [base] ok  error_tecnico      <- 'error_tecnico\nTicket'
  [base] X   solicitud_funcion  <- 'error_tecnico\nTicket'


  [base] X   solicitud_funcion  <- 'error_tecnico\nTicket'
  [base] X   otro               <- 'error_tecnico\nTicket'
  [base] X   otro               <- 'error_tecnico\nTicket'
  [base] accuracy 0.20



## 4. LoRA sobre el base

`LoraConfig` decide dónde van los adaptadores (las cuatro matrices de atención), su rango `r` y la escala `lora_alpha`. `get_peft_model` congela el modelo original y agrega las matrices pequeñas. La línea `print_trainable_parameters` muestra la proporción: con `r=16`, cerca del 0.2 % del total.

El entrenamiento usa el `Trainer` de Hugging Face con next-token prediction sobre la respuesta. 150 pasos con lotes de 4 son unas 19 pasadas sobre los 30 ejemplos: suficiente para un dataset así de pequeño, y una señal de que con pocos datos el riesgo es memorizar.

In [6]:
PASOS = 150
RANGO = 16
SALIDA = "/tmp/lora_facturio"

cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=RANGO,                  # rango de las matrices: 8 a 64 es lo usual
    lora_alpha=2 * RANGO,     # escala del adaptador
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # las matrices de atención
)
modelo = get_peft_model(base, cfg)
modelo.print_trainable_parameters()
modelo.train()

ds = preparar_dataset(tok)
args = TrainingArguments(
    output_dir=SALIDA, max_steps=PASOS, per_device_train_batch_size=4,
    learning_rate=2e-4, logging_steps=25, save_strategy="no", report_to=[],
    fp16=(dtype == torch.float16), use_cpu=(dispositivo == "cpu"),
)
trainer = Trainer(model=modelo, args=args, train_dataset=ds,
                  data_collator=lambda b: collate(b, tok.pad_token_id))
t0 = time.time()
trainer.train()
print(f"entrenamiento: {time.time() - t0:.0f} s, {PASOS} pasos, {len(ENTRENAMIENTO)} ejemplos")

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


/Users/junbarrs/clases_maestria/Arquitectura de Software para sistemas inteligentes/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
25,0.891901
50,0.195825
75,0.016793
100,0.000632
125,0.000340
150,0.000297


entrenamiento: 21 s, 150 pasos, 30 ejemplos


## 5. Base + LoRA sobre los tickets no vistos

Predicción antes de ejecutar: ¿cuánto sube la accuracy con 30 ejemplos? ¿Qué categorías seguirán fallando?

In [7]:
modelo.eval()
print("Base + LoRA, con template:")
acc_lora = evaluar("base+lora", modelo, tok)
modelo.save_pretrained(SALIDA)   # solo el adaptador
peso = sum(p.stat().st_size for p in Path(SALIDA).rglob("*") if p.is_file()) / 1e6
print(f"adaptador guardado en {SALIDA}: {peso:.1f} MB (el base pesa cerca de 1,000 MB)")

Base + LoRA, con template:
  [base+lora] ok  facturacion        <- 'facturacion'
  [base+lora] ok  facturacion        <- 'facturacion'


  [base+lora] ok  acceso             <- 'acceso'
  [base+lora] ok  acceso             <- 'acceso'
  [base+lora] X   error_tecnico      <- 'acceso'


  [base+lora] ok  error_tecnico      <- 'error_tecnico'
  [base+lora] ok  solicitud_funcion  <- 'solicitud_funcion'


  [base+lora] ok  solicitud_funcion  <- 'solicitud_funcion'
  [base+lora] X   otro               <- 'facturacion'
  [base+lora] X   otro               <- 'facturacion'
  [base+lora] accuracy 0.70



adaptador guardado en /tmp/lora_facturio: 8.7 MB (el base pesa cerca de 1,000 MB)


## 6. El instruct oficial, para comparar

El instruct pasó por SFT y preference optimization sobre millones de ejemplos genéricos. No vio las cinco categorías de Facturio. Predicción: ¿le gana o le pierde al base con LoRA en esta tarea?

In [8]:
instruct = AutoModelForCausalLM.from_pretrained(INSTRUCT, dtype=dtype).to(dispositivo).eval()
print("Instruct oficial, con template:")
acc_instruct = evaluar("instruct", instruct, tok)

resumen = {"base_prompt_plano": acc_base, "base_mas_lora": acc_lora, "instruct": acc_instruct,
           "pasos": PASOS, "rango": RANGO, "ejemplos": len(ENTRENAMIENTO)}
print(json.dumps(resumen, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:00<04:25,  1.09it/s]

Loading weights:  92%|█████████▏| 267/290 [00:01<00:00, 358.55it/s]

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 273.26it/s]

Instruct oficial, con template:
  [instruct] X   facturacion        <- 'error_tecnico'


  [instruct] X   facturacion        <- 'error_tecnico'
  [instruct] X   acceso             <- 'error_tecnico'
  [instruct] X   acceso             <- 'error_tecnico'


  [instruct] ok  error_tecnico      <- 'error_tecnico'
  [instruct] ok  error_tecnico      <- 'error_tecnico'
  [instruct] ok  solicitud_funcion  <- 'solicitud_funcion'


  [instruct] X   solicitud_funcion  <- 'error_tecnico'
  [instruct] X   otro               <- 'error_tecnico'
  [instruct] X   otro               <- 'error_tecnico'
  [instruct] accuracy 0.30

{
  "base_prompt_plano": 0.2,
  "base_mas_lora": 0.7,
  "instruct": 0.3,
  "pasos": 150,
  "rango": 16,
  "ejemplos": 30
}


## Lectura

LoRA con 30 ejemplos enseña el formato y las categorías del dominio; en esa tarea supera al instruct genérico. No agrega conocimiento: el modelo sigue sabiendo lo mismo de Facturio que antes.

La condición para justificarlo en un sistema real: medir primero que el prompting con ejemplos de la semana 4 no alcanza la misma accuracy sobre el mismo dataset. Si la alcanza, el adaptador no se justifica, porque cuesta un dataset, un entrenamiento por versión y un modelo distinto que operar.

Para experimentar: cambia `RANGO` (4, 16, 64), `PASOS` (50, 150, 400) o quita categorías del entrenamiento y observa qué pasa con las que faltan.

Referencia: Hu et al. 2021, *LoRA: Low-Rank Adaptation of Large Language Models*. Librería: PEFT de Hugging Face.